# TitleAbstract Reviewer

In this notebook, we will demostrate the use of `TitleAbstractReviewer` agents.

## Setting up the notebook

High-level configs

In [1]:
%reload_ext autoreload
%autoreload 2

from dotenv import load_dotenv

# Load environment variables from .env file. Adjust the path to the .env file as needed.
load_dotenv(dotenv_path='../../.env')

# Enable asyncio in Jupyter
import asyncio
import nest_asyncio

nest_asyncio.apply()

#  Add the package to the path (required if you are running this notebook from the examples folder)
import sys
sys.path.append('../../')


Import required packages

In [2]:
import pandas as pd 

from lattereview.providers import LiteLLMProvider, GoogleProvider
from lattereview.agents import TitleAbstractReviewer
from lattereview.workflows import ReviewWorkflow

## Data

First, let's load a dummy dataset of some articles that we once pooled for a prior systematic review published [here](https://pubmed.ncbi.nlm.nih.gov/36292201/).

In [3]:
data = pd.read_csv('data.csv')
data

,title,abstract,DOI,study_type,clinical_application,organ,modality,cardiovascular,task,deep_learning,external_validation
0,(18)F-FDG PET/CT Uptake Classification in Lymp...,Background Fluorine 18 ((18)F)-fluorodeoxygluc...,10.1148/radiol.2019191114,cross-sectional,diagnosis,lung,pet,0,classification,1,0
1,(18)F-FDG-PET/CT Whole-Body Imaging Lung Tumor...,Under the background of (18)F-FDG-PET/CT multi...,10.1155/2021/8865237,cross-sectional,diagnosis,lung,pet,0,classification,1,0
2,3-D Convolutional Neural Networks for Automati...,Deep two-dimensional (2-D) convolutional neura...,10.1109/jbhi.2018.2879449,cross-sectional,diagnosis,lung,ct,0,detection,1,0
3,3D CNN with Visual Insights for Early Detectio...,The 3D convolutional neural network is able to...,10.1155/2021/6695518,cross-sectional,diagnosis,lung,ct,0,classification,1,0
4,3D deep learning based classification of pulmo...,Classifying ground-glass lung nodules (GGNs) i...,10.1016/j.compmedimag.2020.101814,cross-sectional,diagnosis,lung,ct,0,"segmentation, classification",1,0
...,...,...,...,...,...,...,...,...,...,...,...
973,Vulture-Based AdaBoost-Feedforward Neural Fram...,"In today's scenario, many scientists and medic...",10.1007/s12539-022-00505-3,cross-sectional,diagnosis,lung,xr,0,"segmentation, classification",1,0
974,Wavelet decomposition facilitates training on ...,The adoption of low-dose computed tomography (...,10.1007/s00418-020-01961-y,cross-sectional,diagnosis,lung,ct,0,classification,1,0
975,Weakly unsupervised conditional generative adv...,Because of the rapid spread and wide range of ...,10.1016/j.media.2021.102159,cross-sectional,diagnosis,lung,ct,0,classification,1,0
976,Weakly-supervised lesion analysis with a CNN-b...,Objective.Lesions of COVID-19 can be clearly v...,10.1088/1361-6560/ac4316,cross-sectional,diagnosis,lung,ct,0,classification,1,0


Let's then write some inclusion and exclusion criteria to screen these papers.

In [4]:
inclusion_criteria = "The study must involve CT scans. If multiple modalities are involved, CT scans should be among them."
exclusion_criteria = "The study must not include PET scans as one of its modalities."

## Review

Then, we will define three reviewers (two juniors and one senior) that will go through the dataset and look for those items that meet all the inclusion criteria and does not violate any of the exclusion criteria. Each agent return either of the following scores:

- 1 means absolutely to exclude.
- 2 means better to exclude.
- 3 Not sure if to include or exclude.
- 4 means better to include.
- 5 means absolutely to include.

In [5]:
Agent1 = TitleAbstractReviewer(
    provider=LiteLLMProvider(model="gemini/gemini-3.8-flash"),
    name="Agent1",
    backstory="a PhD researcher in biology and computer science",
    inclusion_criteria = inclusion_criteria,
    exclusion_criteria = exclusion_criteria,        
    max_concurrent_requests=30, 
    model_args={"reasoning_effort": "low"},
)

Agent2 = TitleAbstractReviewer(
    provider=LiteLLMProvider(model="gpt-6-luna"),  
    name="Agent2",
    backstory="a PhD researcher in biology and computer science",
    inclusion_criteria = inclusion_criteria,
    exclusion_criteria = exclusion_criteria,        
    max_concurrent_requests=30, 
)

Agent3 = TitleAbstractReviewer(
    provider=GoogleProvider(model="gemini-3.8-flash"),
    name="Agent3",
    backstory="a senior MD-PhD researcher with years of experience in conducting systematic reviews in radiology and deep learning",
    inclusion_criteria = inclusion_criteria,
    exclusion_criteria = exclusion_criteria,        
    max_concurrent_requests=30, 
    model_args={"thinking_config": {"thinking_level": "high"}},
    additional_context="""
    Two PhD reviewers have already reviewed this article and disagree on how to evaluate it. You can read their evaluation above.
    """
)

Finally, let's create a review workflow and put our agents to work!

In [6]:
def filter_func(row):
    score1 = int(row["round-A_Agent1_output"]["evaluation"])
    score2 = int(row["round-A_Agent2_output"]["evaluation"])
    if score1 != score2:
        if score1 >= 4 and score2 >= 4:
            return False
        if score1 >= 3 or score2 >= 3:
            return True
    elif score1 == score2 == 3:
        return True
    return False

async def review(df, sample_size=None):
    
    title_abs_review = ReviewWorkflow(
        workflow_schema=[
                {
                    "round": 'A',
                    "reviewers": [Agent1, Agent2],
                    "text_inputs": ["title", "abstract"]
                },
                {
                    "round": 'B',
                    "reviewers": [Agent3],
                    "text_inputs": ["title", "abstract", "round-A_Agent1_output", "round-A_Agent2_output"],
                    "filter": filter_func
                }
            ]
        )

    if sample_size:
        df = df.sample(sample_size)

    updated_df = await title_abs_review(df)
    total_cost = title_abs_review.get_total_cost()
    print(f"\n====== Finished reviewing ======\n")
    print(f"\nTotal cost: {total_cost}")
    print("-" * 100)
    return updated_df


updated_df = asyncio.run(review(data, sample_size=10))


====== Starting review round A (1/2) ======

Processing 10 eligible rows


['round: A', 'reviewer_name: Agent1'] -                     2026-09-27 13:26:45: 100%|██████████| 10/10 [00:03<00:00,  2.70it/s]


Reviewer Agent1: 10 successful, 0 failed parses
The following columns are present in the dataframe at the end of Agent1's review in round A: ['title', 'abstract', 'DOI', 'study_type', 'clinical_application', 'organ', 'modality', 'cardiovascular', 'task', 'deep_learning', 'external_validation', 'round-A_Agent1_output', 'round-A_Agent1_reasoning', 'round-A_Agent1_evaluation']


['round: A', 'reviewer_name: Agent2'] -                     2026-09-27 13:26:49: 100%|██████████| 10/10 [00:01<00:00,  5.35it/s]

Reviewer Agent2: 10 successful, 0 failed parses
The following columns are present in the dataframe at the end of Agent2's review in round A: ['title', 'abstract', 'DOI', 'study_type', 'clinical_application', 'organ', 'modality', 'cardiovascular', 'task', 'deep_learning', 'external_validation', 'round-A_Agent1_output', 'round-A_Agent1_reasoning', 'round-A_Agent1_evaluation', 'round-A_Agent2_output', 'round-A_Agent2_reasoning', 'round-A_Agent2_evaluation']

====== Starting review round B (2/2) ======

Skipping review round B - no eligible rows

====== Finished reviewing ======


Total cost: 0.009150699999999998
----------------------------------------------------------------------------------------------------


In [7]:
updated_df

,title,abstract,DOI,study_type,clinical_application,organ,modality,cardiovascular,task,deep_learning,external_validation,round-A_Agent1_output,round-A_Agent1_reasoning,round-A_Agent1_evaluation,round-A_Agent2_output,round-A_Agent2_reasoning,round-A_Agent2_evaluation
806,Pulmonary nodule detection on chest radiograph...,Computer-aided detection (CADe) systems play a...,10.1016/j.artmed.2020.101881,cross-sectional,diagnosis,lung,xr,0,detection,1,0,{'reasoning': 'The study focuses entirely on c...,The study focuses entirely on chest radiograph...,1,{'reasoning': 'The study evaluates pulmonary n...,The study evaluates pulmonary nodule detection...,1
449,Detection of COVID-19 Using Deep Learning Algo...,PURPOSE: To evaluate the performance of a deep...,10.1097/rti.0000000000000559,cross-sectional,diagnosis,lung,xr,0,classification,1,1,{'reasoning': 'The study exclusively evaluates...,The study exclusively evaluates chest radiogra...,1,{'reasoning': 'The study evaluates chest radio...,The study evaluates chest radiographs only and...,1
287,Computer-aided COVID-19 diagnosis and a compar...,BACKGROUND: Coronavirus Disease 2019 (COVID-19...,10.3233/xst-211047,cross-sectional,diagnosis,lung,xr,0,classification,1,0,{'reasoning': 'The study exclusively focuses o...,The study exclusively focuses on chest X-rays ...,1,{'reasoning': 'The study uses chest X-rays onl...,The study uses chest X-rays only and does not ...,1
373,Deep convolutional neural networks for COVID-1...,This article is mainly concerned with COVID-19...,10.1002/jemt.23713,cross-sectional,diagnosis,lung,xr,0,classification,1,0,{'reasoning': 'The study exclusively utilizes ...,The study exclusively utilizes X-ray images fo...,1,{'reasoning': 'The study uses X-ray images and...,The study uses X-ray images and does not invol...,1
345,COVID-view: Diagnosis of COVID-19 using Chest CT,Significant work has been done towards deep le...,10.1109/tvcg.2021.3114851,cross-sectional,diagnosis,lung,ct,0,"segmentation, detection, classification",1,0,{'reasoning': 'The study focuses entirely on c...,The study focuses entirely on chest CT data fo...,5,{'reasoning': 'The study explicitly uses chest...,The study explicitly uses chest CT scans and d...,5
191,Automated Lung Nodule Detection and Classifica...,Lung cancer is one of the major causes of canc...,10.3390/s19173722,cross-sectional,diagnosis,lung,ct,0,"segmentation, classification",1,1,{'reasoning': 'The study evaluates deep learni...,The study evaluates deep learning for lung nod...,5,"{'reasoning': 'The study uses CT scans, includ...","The study uses CT scans, including the LIDC-ID...",5
641,Long-term follow-up of persistent pulmonary pu...,OBJECTIVE: To investigate the natural history ...,10.1007/s00330-019-06344-z,retrospective cohort,diagnosis,lung,ct,0,segmentation,1,0,{'reasoning': 'The study evaluates pulmonary n...,The study evaluates pulmonary nodules using fo...,5,{'reasoning': 'The study uses follow-up CT sca...,The study uses follow-up CT scans and does not...,5
607,INASNET: Automatic identification of coronavir...,Testing is one of the important methodologies ...,10.1016/j.isatra.2022.02.033,cross-sectional,diagnosis,lung,ct,0,classification,1,0,{'reasoning': 'The study focuses exclusively o...,The study focuses exclusively on chest X-ray i...,1,{'reasoning': 'The study uses chest X-ray imag...,The study uses chest X-ray images and does not...,1
446,Detection of COVID-19 from Chest X-ray Images ...,The COVID-19 global pandemic has wreaked havoc...,10.3390/s21175940,cross-sectional,diagnosis,lung,xr,0,classification,1,0,{'reasoning': 'The study exclusively utilizes ...,The study exclusively utilizes chest X-ray ima...,1,{'reasoning': 'The study uses chest X-ray imag...,The study uses chest X-ray images and does not...,1
76,A pilot study using kernelled support tensor m...,We developed a kernelled support tensor machin...,10.1016/j.media.2018.09.004,cross-sectional,treatment,lung,pet,0,classification,1,0,{'reasoning': 'Althou

Now let's print the responses of all three agents for the cases where agent 3 needed to intervene.

In [8]:
# Filter the dataframe for cases where the two junior models (Agent1 and Agent2) disagreed

if "round-B_Agent3_evaluation" in updated_df.columns:   
    disagreed_cases = updated_df[
        updated_df["round-B_Agent3_evaluation"].notna()
    ]

    # Print the responses of all agents for these cases
    for index, row in disagreed_cases.iterrows():
        print(f"Title: {row['title']}")
        print(f"Abstract: {row['abstract']}")
        print(f"Agent1's response: {row['round-A_Agent1_output']}")
        print(f"Agent2's response: {row['round-A_Agent2_output']}")
        print(f"Agent3's response: {row['round-B_Agent3_output']}")
        print("-" * 100)

else:
    print("No cases where Agent3 intervened.")

No cases where Agent3 intervened.
